In [1]:
import os
import pandas as pd
import numpy as np

# fusion method 비교: CUE(CUE, softmax fusion) vs SUM(단순 합산, weight=1,1) vs
# Concat (Early)(cue_concat: 모달리티별 MLP 없이 원본 feature를 concat 후 downstream MLP로 직접 입력) vs
# Concat (Int)(cmfe: 모달리티별 MLP를 거친 embedding을 concat, fusion_policy="concat").
# 네 결과 모두 동일한 5-fold 내부 CV 파이프라인에서 나온 것이며, 유일한 차이는 fusion 방식이다.
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'
cue_concat_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue_concat/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 네 가지 소스: (표시 이름, metrics.csv 루트 경로)
model_sources = {
    'SUM': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'SUM'),
    'Concat (Early)': os.path.join(cue_concat_base, '{dataset}', '{target}', 'v1', 'FP+IMG'),
    'Concat (Int)': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CONCAT'),
    'CUE': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
}

# CUE가 항상 마지막에 오도록 순서를 고정한다.
model_order = ['SUM', 'Concat (Early)', 'Concat (Int)', 'CUE']


In [2]:
def load_metric(model_dir_template, dataset, target):
    """cue/cue_concat의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다.
    두 저장소 모두 동일한 aggregate_metrics 함수를 사용하므로 스키마가 동일하다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다.")


✅ 데이터 로드 완료: 32개의 항목이 수집되었습니다.


In [3]:
# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능 (아래 build_rank_matrix 셀에서 계속 사용).
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}
METRIC_HEADERS = {'rmse': 'RMSE( ↓ )', 'r2': 'R2( ↑ )', 'pcc': 'PCC( ↑ )'}


def build_metric_table(df_all, dataset, target, metrics=metrics_of_interest, model_order=model_order):
    """지정한 dataset/target에 대해 Model별 metric 값('mean ± std')을 담은 표를 만든다 (순위 없음).

    Returns:
        Optional[pd.DataFrame]: 컬럼이 Model | RMSE( ↓ ) | R2( ↑ ) | PCC( ↑ ) 순서인 표.
            각 값은 '{mean:.3f} ± {std:.3f}' 문자열이다.
            해당 dataset/target에 데이터가 하나도 없으면 None.
    """
    df_sub = df_all[(df_all['Dataset'] == dataset) & (df_all['Target'] == target)]
    df_sub = df_sub.set_index('Model').reindex(model_order)
    df_sub = df_sub.dropna(subset=[m for m in metrics if m in df_sub.columns], how='all')
    if df_sub.empty:
        return None

    table = pd.DataFrame(index=df_sub.index)
    for m in metrics:
        if m not in df_sub.columns:
            continue
        header = METRIC_HEADERS.get(m, m.upper())
        if f'{m}_std' in df_sub.columns:
            table[header] = [f"{v:.3f} ± {s:.3f}" for v, s in zip(df_sub[m], df_sub[f'{m}_std'])]
        else:
            table[header] = df_sub[m].round(3)

    return table.reset_index()  # 'Model'을 첫 컬럼으로 복원


table_save_dir = os.path.join(cue_base, 'table', 'fusion_comparison_v2')
os.makedirs(table_save_dir, exist_ok=True)

# DIS -> dis(dissociation), INH -> inh(inhibition)
dataset_filenames = {'DIS': 'dis.csv', 'INH': 'inh.csv'}

metric_tables = {}
for d_name in datasets:
    dataset_tables = []
    for target in targets:
        table = build_metric_table(df_total, d_name, target)
        if table is None:
            continue
        metric_tables[(d_name, target)] = table

        print(f"\n=== [{d_name} / {target}] ===")
        print(table.to_string(index=False))

        # 맨 처음 컬럼에 Target을 추가한 뒤(targets 순서 = PI, Ssel, WS2, RS2) 이어 붙인다.
        table_with_target = table.copy()
        table_with_target.insert(0, 'Target', target)
        dataset_tables.append(table_with_target)

    if not dataset_tables:
        continue

    dataset_table = pd.concat(dataset_tables, ignore_index=True)
    save_path = os.path.join(table_save_dir, dataset_filenames[d_name])
    dataset_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
    print(f"\n✅ 저장 완료: {save_path}")


=== [Kd / PI] ===
         Model     RMSE( ↓ )       R2( ↑ )      PCC( ↑ )
           SUM 0.242 ± 0.010 0.124 ± 0.057 0.363 ± 0.096
Concat (Early) 0.237 ± 0.012 0.159 ± 0.064 0.419 ± 0.077
  Concat (Int) 0.242 ± 0.011 0.119 ± 0.060 0.361 ± 0.093
           CUE 0.242 ± 0.010 0.125 ± 0.055 0.366 ± 0.092

=== [Kd / Ssel] ===
         Model     RMSE( ↓ )       R2( ↑ )      PCC( ↑ )
           SUM 0.904 ± 0.074 0.046 ± 0.119 0.300 ± 0.109
Concat (Early) 0.893 ± 0.060 0.071 ± 0.079 0.334 ± 0.067
  Concat (Int) 0.899 ± 0.072 0.056 ± 0.107 0.302 ± 0.105
           CUE 0.895 ± 0.068 0.065 ± 0.107 0.314 ± 0.104

=== [Kd / WS(alpha=2)] ===
         Model     RMSE( ↓ )       R2( ↑ )      PCC( ↑ )
           SUM 0.283 ± 0.013 0.211 ± 0.076 0.487 ± 0.059
Concat (Early) 0.293 ± 0.024 0.154 ± 0.127 0.459 ± 0.059
  Concat (Int) 0.284 ± 0.014 0.206 ± 0.067 0.482 ± 0.057
           CUE 0.282 ± 0.014 0.221 ± 0.061 0.490 ± 0.052

=== [Kd / RS(k=2)] ===
         Model     RMSE( ↓ )        R2( ↑ )      PCC(

In [4]:
# metric별 rank 행렬: row = Dataset-Target(8개) + 마지막 'avg' 행, column = model_order(SUM/Concat (Early)/Concat (Int)/CUE).
def build_rank_matrix(df_all, metric, model_order=model_order):
    """지정한 metric에 대해 8개의 (Dataset, Target) 조합별로 model_order 각 모델의 순위를 담은
    행렬을 만든다. 마지막 'avg' 행은 각 모델의 8개 행별 순위 평균이다.

    Returns:
        pd.DataFrame: 컬럼이 'Dataset-Target', SUM, Concat (Early), Concat (Int), CUE 순인 표.
    """
    rows = {}
    for d_name in datasets:
        for target in targets:
            df_sub = df_all[(df_all['Dataset'] == d_name) & (df_all['Target'] == target)]
            df_sub = df_sub.set_index('Model').reindex(model_order)
            if metric not in df_sub.columns or df_sub[metric].isna().all():
                continue
            ranks = df_sub[metric].rank(ascending=LOWER_IS_BETTER.get(metric, True), method='min')
            rows[f'{d_name}-{target}'] = ranks

    rank_df = pd.DataFrame(rows).T
    rank_df = rank_df.reindex(columns=model_order)
    rank_df.loc['avg'] = rank_df.mean(axis=0).round(1)

    return rank_df.reset_index().rename(columns={'index': 'Dataset-Target'})


for metric in metrics_of_interest:
    rank_table = build_rank_matrix(df_total, metric)

    print(f"\n=== Rank table ({metric}) ===")
    print(rank_table.to_string(index=False))

    save_path = os.path.join(table_save_dir, f'{metric}.csv')
    rank_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
    print(f"✅ 저장 완료: {save_path}")


=== Rank table (rmse) ===
Dataset-Target  SUM  Concat (Early)  Concat (Int)  CUE
         Kd-PI  3.0             1.0           4.0  2.0
       Kd-Ssel  4.0             1.0           3.0  2.0
Kd-WS(alpha=2)  2.0             4.0           3.0  1.0
    Kd-RS(k=2)  3.0             4.0           1.0  2.0
         Ki-PI  2.0             3.0           4.0  1.0
       Ki-Ssel  2.0             4.0           1.0  3.0
Ki-WS(alpha=2)  3.0             4.0           1.0  2.0
    Ki-RS(k=2)  2.0             4.0           1.0  3.0
           avg  2.6             3.1           2.2  2.0
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/table/fusion_comparison_v2/rmse.csv

=== Rank table (r2) ===
Dataset-Target  SUM  Concat (Early)  Concat (Int)  CUE
         Kd-PI  3.0             1.0           4.0  2.0
       Kd-Ssel  4.0             1.0           3.0  2.0
Kd-WS(alpha=2)  2.0             4.0           3.0  1.0
    Kd-RS(k=2)  3.0             4.0           1.0  2.0
         Ki-PI  2.0            